# Importer des fichiers CSV dans une base avec SQLAlchemy Core

On importe ici dans une base SQLite les données d'une bibliothèque, écrites dans des fichiers CSV, avec SQLAlchemy Core : décrire les tables des livres, des membres et de leurs emprunts, importer les fichiers, interroger la base, exporter les emprunts en CSV, puis les analyser avec pandas. Il suppose la partie Core du TP « Prise en main de SQLAlchemy ».

Les étapes s'enchaînent : chaque solution définit ce que la suite utilise. Si vous êtes bloqué, exécutez la solution avant de passer à la suite.

## Mise en place

Les trois cellules suivantes écrivent les données de la bibliothèque, une table par fichier CSV : les livres (`livres.csv`), les membres (`membres.csv`), et les emprunts datés (`emprunts.csv`), qui désignent livre et membre par leur titre et leur nom ; certains sont rendus (ils ont une date de `retour`). La suivante crée le moteur, sur le fichier `bibliotheque.db`.

In [ ]:
%%writefile livres.csv
titre,auteur
Vingt Mille Lieues sous les mers,Jules Verne
Voyage au centre de la Terre,Jules Verne
De la Terre à la Lune,Jules Verne
Le Tour du monde en quatre-vingts jours,Jules Verne
Michel Strogoff,Jules Verne
The Hound of the Baskervilles,Arthur Conan Doyle
"Arsène Lupin, gentleman-cambrioleur",Maurice Leblanc

In [ ]:
%%writefile membres.csv
nom
John Doe
Jane Doe
Sherlock Holmes
Arsène Lupin

In [ ]:
%%writefile emprunts.csv
livre,membre,date,retour
Voyage au centre de la Terre,Jane Doe,2026-01-03,2026-01-24
Le Tour du monde en quatre-vingts jours,Sherlock Holmes,2026-01-05,2026-01-19
Michel Strogoff,Sherlock Holmes,2026-01-07,
"Arsène Lupin, gentleman-cambrioleur",Jane Doe,2026-01-20,2026-02-01
The Hound of the Baskervilles,Sherlock Holmes,2026-02-02,
Voyage au centre de la Terre,John Doe,2026-02-10,
Le Tour du monde en quatre-vingts jours,Sherlock Holmes,2026-02-20,2026-03-06

In [ ]:
from sqlalchemy import create_engine

engine = create_engine("sqlite:///bibliotheque.db")

## 1. Décrire les tables

Un membre emprunte plusieurs livres, et un livre est emprunté par plusieurs membres au fil du temps : la table des emprunts relie les deux par deux clefs étrangères, et porte en plus ses propres informations, la date de l'emprunt et celle du retour. Avec un objet `MetaData` nommé `metadata`, décrivez trois tables :

- `livres` (variable `table_livres`) : `id`, clef primaire entière ; `titre`, chaîne obligatoire et unique ; `auteur`, chaîne obligatoire ;
- `membres` (variable `table_membres`) : `id` ; `nom`, chaîne obligatoire et unique ;
- `emprunts` (variable `table_emprunts`) : `id` ; `id_livre` et `id_membre`, obligatoires, clefs étrangères vers les colonnes `id` de `livres` et de `membres` ; `date_emprunt`, date obligatoire ; `date_retour`, date facultative (un emprunt en cours n'en a pas).

Supprimez les tables qu'une exécution précédente aurait laissées, puis créez celles-ci. Pourquoi un emprunt a-t-il son propre `id`, plutôt qu'une clef primaire formée de `id_livre` et `id_membre` ?

**Documentation utile**

- [Contraintes](https://docs.sqlalchemy.org/en/21/tutorial/metadata.html#declaring-simple-constraints)
- [`Date`](https://docs.sqlalchemy.org/en/21/core/type_basics.html#sqlalchemy.types.Date)

In [ ]:
# Votre code ici

### Conseils supplémentaires

« Obligatoire » se dit `nullable=False`, « unique » `unique=True`, et une colonne `Date` reçoit des objets `datetime.date`. Une colonne qui porte une `ForeignKey("livres.id")` en prend le type : inutile de le répéter. `metadata.drop_all(engine)` puis `metadata.create_all(engine)` suppriment puis créent toutes les tables décrites.

```python
from sqlalchemy import Column, Date, ForeignKey, Integer, MetaData, String, Table

metadata = MetaData()

table_livres = Table(
  "livres",
  metadata,
  Column("id", Integer, primary_key=True),
  Column("titre", String, nullable=False, unique=True),
  Column("auteur", String, nullable=False),
)

table_membres = Table(...)

table_emprunts = Table(
  "emprunts",
  metadata,
  Column("id", Integer, primary_key=True),
  Column("id_livre", ForeignKey(...), nullable=False),
  ...,
)

metadata.drop_all(engine)
metadata.create_all(engine)
```

### Solution

In [ ]:
from sqlalchemy import Column, Date, ForeignKey, Integer, MetaData, String, Table

metadata = MetaData()

table_livres = Table(
  "livres",
  metadata,
  Column("id", Integer, primary_key=True),
  Column("titre", String, nullable=False, unique=True),
  Column("auteur", String, nullable=False),
)

table_membres = Table(
  "membres",
  metadata,
  Column("id", Integer, primary_key=True),
  Column("nom", String, nullable=False, unique=True),
)

table_emprunts = Table(
  "emprunts",
  metadata,
  Column("id", Integer, primary_key=True),
  Column("id_livre", ForeignKey("livres.id"), nullable=False),
  Column("id_membre", ForeignKey("membres.id"), nullable=False),
  Column("date_emprunt", Date, nullable=False),
  Column("date_retour", Date),
)

metadata.drop_all(engine)
metadata.create_all(engine)

# Un même membre peut emprunter plusieurs fois le même livre (dans les données,
# Sherlock Holmes et Le Tour du monde en quatre-vingts jours) : le couple
# (id_livre, id_membre) n'identifie pas un emprunt, et ne peut donc pas servir de
# clef primaire. D'où un id propre à chaque emprunt.

## 2. Importer les fichiers CSV

Complétez la fonction `importer_csv(dossier, conn)`, qui lit les fichiers `livres.csv`, `membres.csv` et `emprunts.csv` du dossier `dossier`, et insère leurs lignes dans les trois tables, une instruction `insert` par table. Les emprunts désignent livres et membres par leur titre et leur nom, quand la table `emprunts` attend leurs `id` : relisez-les dans la base une fois les livres et les membres insérés. Appelez-la ensuite sur le dossier courant (`Path(".")`), en une seule transaction, puis affichez le nombre de lignes de chaque table.

**Documentation utile**

- [`csv.DictReader`](https://docs.python.org/fr/3/library/csv.html#csv.DictReader)
- [`date.fromisoformat`](https://docs.python.org/fr/3/library/datetime.html#datetime.date.fromisoformat)

In [ ]:
from pathlib import Path

from sqlalchemy import Connection


def importer_csv(dossier: Path, conn: Connection) -> None:
  """Insère dans la base les livres, membres et emprunts des CSV de dossier."""
  ...

In [ ]:
# Votre code ici

### Conseils supplémentaires

`csv.DictReader(f)` lit un fichier CSV ligne à ligne, chaque ligne en dictionnaire dont les clefs sont les noms de colonnes de l'en-tête ; ouvrez le fichier avec `newline=""`, comme le demande le module `csv`. Le titre qui contient une virgule est entre guillemets dans le fichier : `csv` les retire. Les en-têtes de `livres.csv` et de `membres.csv` sont les noms des colonnes : leurs lignes s'insèrent telles quelles, toutes d'un coup, avec `conn.execute(insert(table_livres), lignes)`. Pour les emprunts, `dict(conn.execute(select(table_livres.c.titre, table_livres.c.id)).all())` associe à chaque titre son `id`. Toutes les valeurs lues sont des chaînes : `date.fromisoformat` convertit les dates, et `retour` est une chaîne vide pour un emprunt en cours. Vous devez obtenir 7 livres, 4 membres et 7 emprunts. Réexécuté, l'import lève une `IntegrityError`, puisque titres et noms sont uniques : réexécutez d'abord la création des tables.

```python
import csv
from datetime import date

from sqlalchemy import insert, select


def lire_csv(chemin: Path) -> list[dict[str, str]]:
  with open(chemin, newline="", encoding="utf-8") as f:
    return list(csv.DictReader(f))


def lire_date(texte: str) -> date | None:
  return date.fromisoformat(texte) if texte else None


def importer_csv(dossier: Path, conn: Connection) -> None:
  conn.execute(insert(table_livres), lire_csv(dossier / "livres.csv"))
  conn.execute(...)
  id_livres = dict(conn.execute(select(table_livres.c.titre, table_livres.c.id)).all())
  id_membres = ...
  conn.execute(
    insert(table_emprunts),
    [
      {
        "id_livre": id_livres[ligne["livre"]],
        "id_membre": ...,
        "date_emprunt": ...,
        "date_retour": ...,
      }
      for ligne in lire_csv(dossier / "emprunts.csv")
    ],
  )


with engine.begin() as conn:
  importer_csv(Path("."), conn)

with engine.connect() as conn:
  for table in (table_livres, table_membres, table_emprunts):
    print(table.name, len(conn.execute(select(table)).all()))
```

### Solution

In [ ]:
import csv
from datetime import date

from sqlalchemy import insert, select


def lire_csv(chemin: Path) -> list[dict[str, str]]:
  with open(chemin, newline="", encoding="utf-8") as f:
    return list(csv.DictReader(f))


def lire_date(texte: str) -> date | None:
  return date.fromisoformat(texte) if texte else None


def importer_csv(dossier: Path, conn: Connection) -> None:
  conn.execute(insert(table_livres), lire_csv(dossier / "livres.csv"))
  conn.execute(insert(table_membres), lire_csv(dossier / "membres.csv"))
  id_livres = dict(conn.execute(select(table_livres.c.titre, table_livres.c.id)).all())
  id_membres = dict(conn.execute(select(table_membres.c.nom, table_membres.c.id)).all())
  conn.execute(
    insert(table_emprunts),
    [
      {
        "id_livre": id_livres[ligne["livre"]],
        "id_membre": id_membres[ligne["membre"]],
        "date_emprunt": lire_date(ligne["date"]),
        "date_retour": lire_date(ligne["retour"]),
      }
      for ligne in lire_csv(dossier / "emprunts.csv")
    ],
  )


with engine.begin() as conn:
  importer_csv(Path("."), conn)

with engine.connect() as conn:
  for table in (table_livres, table_membres, table_emprunts):
    print(table.name, len(conn.execute(select(table)).all()))

# Pour les livres et les membres, pandas aurait suffi :
# pd.read_csv("livres.csv").to_sql("livres", engine, if_exists="append", index=False)
# Les emprunts, eux, demandent de retrouver les id des livres et des membres.

## 3. Interroger la base

Chaque requête de cette partie s'écrit avec `select`, et s'exécute dans un bloc `with engine.connect() as conn:`.

### Les livres empruntés par Sherlock Holmes

Affichez le titre des livres que Sherlock Holmes a empruntés, avec la date de chaque emprunt, triés par date.

**Documentation utile**

- [Jointures](https://docs.sqlalchemy.org/en/21/tutorial/data_select.html#explicit-from-clauses-and-joins)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Partez de la table des emprunts, `select_from(table_emprunts)`, puis ajoutez chaque autre table par un `join(...)` : SQLAlchemy trouve la condition de jointure grâce à la clef étrangère. Filtrez sur `table_membres.c.nom` et triez par `table_emprunts.c.date_emprunt` avec `order_by`. Chaque ligne du résultat a un attribut par colonne du `select`. Vous devez obtenir quatre emprunts, dont deux du même livre, du 5 janvier au 20 février 2026.

```python
with engine.connect() as conn:
  requete = (
    select(table_emprunts.c.date_emprunt, table_livres.c.titre)
    .select_from(table_emprunts)
    .join(table_livres)
    .join(...)
    .where(...)
    .order_by(...)
  )
  for ligne in conn.execute(requete):
    print(ligne.date_emprunt, ligne.titre)
```

#### Solution

In [ ]:
with engine.connect() as conn:
  requete = (
    select(table_emprunts.c.date_emprunt, table_livres.c.titre)
    .select_from(table_emprunts)
    .join(table_livres)
    .join(table_membres)
    .where(table_membres.c.nom == "Sherlock Holmes")
    .order_by(table_emprunts.c.date_emprunt)
  )
  for ligne in conn.execute(requete):
    print(ligne.date_emprunt, ligne.titre)

### Les emprunts en cours

Affichez les emprunts en cours, ceux qui n'ont pas de date de retour, par date d'emprunt : la date, le titre du livre et le nom du membre.

**Documentation utile**

- [`is_`](https://docs.sqlalchemy.org/en/21/core/sqlelement.html#sqlalchemy.sql.expression.ColumnOperators.is_)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

En SQL, l'absence de valeur se teste avec `IS NULL`, que SQLAlchemy écrit `table_emprunts.c.date_retour.is_(None)` (et non `== None`, que les vérificateurs de style signalent). La requête joint les trois tables, comme la précédente. Vous devez obtenir 3 emprunts.

```python
with engine.connect() as conn:
  en_cours = ...
  requete = (
    select(table_emprunts.c.date_emprunt, table_livres.c.titre, table_membres.c.nom)
    .select_from(table_emprunts)
    .join(...)
    .join(...)
    .where(...)
    .order_by(...)
  )
  for ligne in conn.execute(requete):
    print(ligne.date_emprunt, ligne.titre, ligne.nom)
```

#### Solution

In [ ]:
with engine.connect() as conn:
  en_cours = table_emprunts.c.date_retour.is_(None)
  requete = (
    select(table_emprunts.c.date_emprunt, table_livres.c.titre, table_membres.c.nom)
    .select_from(table_emprunts)
    .join(table_livres)
    .join(table_membres)
    .where(en_cours)
    .order_by(table_emprunts.c.date_emprunt)
  )
  for ligne in conn.execute(requete):
    print(ligne.date_emprunt, ligne.titre, ligne.nom)

### Les livres disponibles

Affichez, par ordre alphabétique, le titre des livres disponibles : ceux qui n'ont aucun emprunt en cours.

**Documentation utile**

- [`not_in`](https://docs.sqlalchemy.org/en/21/core/sqlelement.html#sqlalchemy.sql.expression.ColumnOperators.not_in)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

`colonne.not_in(sous_requete)` garde les lignes dont la valeur n'est pas dans le résultat d'un autre `select` (en SQL, `NOT IN (SELECT ...)`) : ici, les livres dont l'`id` n'est pas parmi les `id_livre` des emprunts en cours. La condition « en cours » est celle de l'exercice précédent. Attention : un livre rendu peut avoir été emprunté de nouveau. `conn.scalars(...)` renvoie la première colonne de chaque ligne. Vous devez obtenir 4 livres.

```python
with engine.connect() as conn:
  en_cours = table_emprunts.c.date_retour.is_(None)
  empruntes = select(...).where(en_cours)
  requete = select(table_livres.c.titre).where(...).order_by(table_livres.c.titre)
  for titre in conn.scalars(requete):
    print(titre)
```

#### Solution

In [ ]:
with engine.connect() as conn:
  en_cours = table_emprunts.c.date_retour.is_(None)
  empruntes = select(table_emprunts.c.id_livre).where(en_cours)
  requete = (
    select(table_livres.c.titre)
    .where(table_livres.c.id.not_in(empruntes))
    .order_by(table_livres.c.titre)
  )
  for titre in conn.scalars(requete):
    print(titre)

# Voyage au centre de la Terre n'y est pas : rendu par Jane Doe, il a été emprunté
# de nouveau, par John Doe. Un livre est disponible s'il n'a aucun emprunt en
# cours, et non s'il a un emprunt rendu.

## 4. Exporter les emprunts en CSV

Complétez la fonction `exporter_csv(chemin, conn)`, qui écrit tous les emprunts, par date d'emprunt, dans le fichier CSV `chemin` : une ligne par emprunt, avec les colonnes `livre`, `membre`, `date_emprunt` et `date_retour` (vide pour un emprunt en cours). Appelez-la pour écrire `export.csv`, puis affichez le contenu du fichier.

**Documentation utile**

- [`csv.DictWriter`](https://docs.python.org/fr/3/library/csv.html#csv.DictWriter)
- [`ColumnElement.label`](https://docs.sqlalchemy.org/en/21/core/sqlelement.html#sqlalchemy.sql.expression.ColumnElement.label)
- [`Result.mappings`](https://docs.sqlalchemy.org/en/21/core/connections.html#sqlalchemy.engine.Result.mappings)

In [ ]:
def exporter_csv(chemin: str, conn: Connection) -> None:
  """Écrit les emprunts de la base dans le fichier CSV chemin."""
  ...

In [ ]:
# Votre code ici

### Conseils supplémentaires

`.label("livre")` nomme une colonne du `select`, et `conn.execute(requete).mappings()` rend chaque ligne comme un dictionnaire dont les clefs sont les noms des colonnes. `csv.DictWriter(f, fieldnames=[...])` écrit de tels dictionnaires : `writeheader()` écrit l'en-tête, puis `writerow(...)` chaque ligne ; ouvrez le fichier avec `newline=""`. `csv` écrit une date au format `2026-01-03`, et `None` comme une case vide. Le fichier doit avoir 8 lignes : l'en-tête et 7 emprunts.

```python
def exporter_csv(chemin: str, conn: Connection) -> None:
  requete = (
    select(
      table_livres.c.titre.label("livre"),
      ...,
    )
    .select_from(table_emprunts)
    .join(...)
    .join(...)
    .order_by(...)
  )
  with open(chemin, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
      f, fieldnames=["livre", "membre", "date_emprunt", "date_retour"]
    )
    writer.writeheader()
    for ligne in conn.execute(requete).mappings():
      ...


with engine.connect() as conn:
  exporter_csv("export.csv", conn)

with open("export.csv", encoding="utf-8") as f:
  print(f.read())
```

### Solution

In [ ]:
def exporter_csv(chemin: str, conn: Connection) -> None:
  requete = (
    select(
      table_livres.c.titre.label("livre"),
      table_membres.c.nom.label("membre"),
      table_emprunts.c.date_emprunt,
      table_emprunts.c.date_retour,
    )
    .select_from(table_emprunts)
    .join(table_livres)
    .join(table_membres)
    .order_by(table_emprunts.c.date_emprunt)
  )
  with open(chemin, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
      f, fieldnames=["livre", "membre", "date_emprunt", "date_retour"]
    )
    writer.writeheader()
    for ligne in conn.execute(requete).mappings():
      writer.writerow(ligne)


with engine.connect() as conn:
  exporter_csv("export.csv", conn)

with open("export.csv", encoding="utf-8") as f:
  print(f.read())

## 5. Analyser les emprunts avec pandas

Avec `pd.read_sql`, chargez dans un `DataFrame` `emprunts` le résultat d'une requête `select` qui donne, pour chaque emprunt, le nom du membre (colonne `membre`), le titre du livre (colonne `livre`) et la date d'emprunt. Affichez ensuite le nombre d'emprunts de chaque membre, du plus grand au plus petit.

**Documentation utile**

- [`Series.value_counts`](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`pd.read_sql(requete, engine)` exécute la requête et renvoie un `DataFrame`, une colonne par élément du `select`, nommée par son `label`. La requête joint les trois tables, comme celle de l'export. `value_counts()` compte les valeurs d'une colonne, de la plus fréquente à la moins fréquente. Vous devez obtenir Sherlock Holmes 4, Jane Doe 2 et John Doe 1.

```python
import pandas as pd

requete = (
  select(table_membres.c.nom.label("membre"), ..., table_emprunts.c.date_emprunt)
  .select_from(table_emprunts)
  .join(...)
  .join(...)
)
emprunts = pd.read_sql(requete, engine)
print(...)
```

### Solution

In [ ]:
import pandas as pd

requete = (
  select(
    table_membres.c.nom.label("membre"),
    table_livres.c.titre.label("livre"),
    table_emprunts.c.date_emprunt,
  )
  .select_from(table_emprunts)
  .join(table_membres)
  .join(table_livres)
)
emprunts = pd.read_sql(requete, engine)
print(emprunts["membre"].value_counts())